# CH10 — Diccionarios: construye tu propio `dict` con una lista

Material del curso basado en Goodrich, Tamassia & Goldwasser — Capítulo 10.

**Objetivo:** entender qué hace cada método de un diccionario de Python y construir, **desde cero**, una clase que se comporte igual guardando la información en una **lista**.

**Contenido**

1. ¿Qué es un diccionario (mapa)?
2. Los métodos de un diccionario, uno por uno
3. La idea: guardar las entradas en una lista no ordenada
4. **Para la clase:** implementa `UnsortedTableMap`
5. Pruebas

## 1. ¿Qué es un diccionario (mapa)?

- Un **diccionario** (en Goodrich, un **mapa**) guarda una colección de **entradas** `(clave, valor)`.
- Las **claves son únicas**: cada clave aparece a lo sumo una vez. Asignar un valor a una clave que ya existe **reemplaza** el valor anterior.
- Se accede a un valor **por su clave**, no por su posición. Es como una lista donde el "índice" puede ser cualquier objeto: un nombre, una cédula, una palabra.

**Ejemplos:** un directorio telefónico (nombre → número), un registro de notas (código → nota), un contador de palabras (palabra → número de apariciones).

## 2. Los métodos de un diccionario

### Crear un diccionario

| Forma | Ejemplo |
|---|---|
| Literal | `d = {'ana': 4.5, 'luis': 3.8}` |
| Constructor con pares | `d = dict([('ana', 4.5), ('luis', 3.8)])` |
| Vacío | `d = {}` o `d = dict()` |

### Acceso, inserción y eliminación

| Método | Qué hace | Si la clave no existe |
|---|---|---|
| `d[k]` | Retorna el valor de `k` | Lanza `KeyError` |
| `d[k] = v` | Asigna `v` a `k`: inserta si es nueva, **reemplaza** si ya existe | La crea |
| `del d[k]` | Elimina la entrada de `k` | Lanza `KeyError` |
| `d.get(k, default=None)` | Retorna el valor de `k` | Retorna `default` |
| `d.setdefault(k, default=None)` | Si `k` existe retorna su valor; si no, **inserta** `k: default` y retorna `default` | La crea con `default` |
| `d.pop(k)` | Elimina y **retorna** el valor de `k` | Lanza `KeyError` |
| `d.pop(k, default)` | Elimina y retorna el valor de `k` | Retorna `default` |
| `d.popitem()` | Elimina y retorna el par `(k, v)` **insertado de último** | Lanza `KeyError` si está vacío |
| `d.clear()` | Elimina todas las entradas | — |

### Consultas y recorridos

| Método | Qué hace |
|---|---|
| `len(d)` | Número de entradas |
| `k in d` | `True` si existe la **clave** `k` (no busca en los valores) |
| `for k in d:` | Recorre las **claves**, en orden de inserción |
| `d.keys()` | Las claves |
| `d.values()` | Los valores |
| `d.items()` | Los pares `(k, v)`, ideal para `for k, v in d.items():` |

### Combinar y comparar

| Método | Qué hace |
|---|---|
| `d.update(otro)` | Inserta o reemplaza todas las entradas de `otro` |
| `d == otro` | `True` si tienen exactamente los mismos pares, **sin importar el orden** |

**Detalles importantes:**
- Reemplazar el valor de una clave que ya existe **no** cambia su posición en el orden.
- Las claves deben ser **inmutables** (`int`, `str`, `tuple`...). Una `list` no puede ser clave.

In [1]:
# --- Acceso, inserción y eliminación con el dict de Python ---
notas = {'ana': 4.5, 'luis': 3.8}

notas['maria'] = 4.1              # inserta
notas['luis'] = 4.0               # reemplaza (luis conserva su posición)
print(notas)                      # {'ana': 4.5, 'luis': 4.0, 'maria': 4.1}

print(notas['ana'])               # 4.5
print(notas.get('pedro'))         # None
print(notas.get('pedro', 0.0))    # 0.0

print(notas.setdefault('pedro', 3.0))  # 3.0  -> la inserta
print(notas.setdefault('ana', 1.0))    # 4.5  -> ya existía, no cambia

print(notas.pop('pedro'))              # 3.0
print(notas.pop('pedro', 'no está'))   # no está
del notas['maria']
print(notas)                      # {'ana': 4.5, 'luis': 4.0}

try:
    notas['zoe']
except KeyError as e:
    print('KeyError:', e)

{'ana': 4.5, 'luis': 4.0, 'maria': 4.1}
4.5
None
0.0
3.0
4.5
3.0
no está
{'ana': 4.5, 'luis': 4.0}
KeyError: 'zoe'


In [ ]:
# --- Consultas, recorridos y combinación con el dict de Python ---
d = {'a': 1, 'b': 2, 'c': 3}

print(len(d))                     # 3
print('b' in d, 2 in d)           # True False   (in busca claves, no valores)
print(list(d.keys()), list(d.values()), list(d.items()))

for k, v in d.items():
    print(k, '->', v)

d['z'] = 26
print(d.popitem())                # ('z', 26)  -> el último insertado

d.update({'a': 100, 'e': 5})
print(d)                          # {'a': 100, 'b': 2, 'c': 3, 'e': 5}
print({'a': 1, 'b': 2} == {'b': 2, 'a': 1})   # True: el orden no importa

d.clear()
print(d)                          # {}

## 3. La idea: guardar las entradas en una lista no ordenada

Vamos a construir nuestro propio diccionario **sin usar `dict`**. Toda la información se guarda en una lista de Python, `self._table`, donde cada elemento es una **entrada** `[clave, valor]`:

```
self._table = [ ['ana', 4.5], ['luis', 4.0], ['maria', 4.1] ]
```

Cada entrada es una **lista** de dos posiciones (y no una tupla) para poder **cambiar el valor** cuando se reasigna una clave: `entrada[1] = nuevo_valor`.

Las entradas se guardan en el orden en que llegan (se agregan al final con `append`), así que la lista queda en **orden de inserción**, igual que un `dict`. Pero **no están ordenadas por clave**: por eso se llama *unsorted table*.

Como no hay ningún orden por clave, **para encontrar una clave hay que recorrer la lista** comparando una por una:

| Operación | Idea | Costo |
|---|---|---|
| `M[k]` | Recorrer hasta encontrar la entrada con clave `k` | $O(n)$ |
| `M[k] = v` | Recorrer: si la clave ya está, reemplazar el valor; si no, `append` al final | $O(n)$ |
| `del M[k]` | Recorrer hasta encontrar el índice `j` de la clave y hacer `pop(j)` | $O(n)$ |
| `len(M)` | `len(self._table)` | $O(1)$ |

Ojo con `M[k] = v`: aunque `append` es $O(1)$, **primero hay que buscar** si la clave ya existe, porque las claves deben ser únicas.

## 4. Para la clase: implementa `UnsortedTableMap`

`__init__` y `__repr__` ya están dados. Completa cada método marcado con `pass`, en este orden (los últimos pueden reutilizar los primeros).

En esta primera tanda solo se implementan los **métodos especiales**. Los métodos con nombre de la sección 2 (`get`, `pop`, `keys`, `items`, `update`, ...) quedan para una segunda tanda.

- `_buscar(k)`: método **auxiliar** (no es parte del diccionario). Recorre `self._table` y retorna el **índice** `j` de la entrada cuya clave es `k`, o `-1` si no existe. Todos los demás métodos pueden apoyarse en él.
- `__len__()`: número de entradas.
- `__getitem__(k)`: retorna el valor de `k`. Si no existe, `raise KeyError(k)`.
- `__setitem__(k, v)`: si `k` ya existe, reemplaza su valor; si no, agrega `[k, v]` al final.
- `__delitem__(k)`: elimina la entrada de `k` con `pop(j)`. Si no existe, `raise KeyError(k)`.
- `__contains__(k)`: `True` si `k` es una clave. Se usa como `k in M`.
- `__iter__()`: **generador** que hace `yield` de cada **clave**. Se usa en `for k in M`.
- `__eq__(otro)`: `True` si `otro` tiene el mismo tamaño y, para cada clave `k` propia, `k in otro` y `otro[k]` es igual al valor propio. Así el orden no importa.

**Regla:** no uses `dict` dentro de la clase. Todo se guarda en `self._table`.

In [1]:
class UnsortedTableMap:
    # Diccionario implementado desde cero con una lista no ordenada de entradas [clave, valor].

    def __init__(self):
        # Crea un diccionario vacío.
        self._table = []                  # lista de entradas [clave, valor]

    # ---------- auxiliar ----------
    def _buscar(self, k):
        # Retorna el índice de la entrada con clave k, o -1 si no existe.
        for i in range(len(self._table)):
            if self._table[i][0] == k:
                return i
        return -1

    # ---------- núcleo: métodos especiales ----------
    def __len__(self):
        # len(M)
        return len(self._table)

    def __getitem__(self, k):
        # M[k]  (KeyError si no existe)
        i = self._buscar(k)
        if i == -1:
            raise KeyError(k)
        return self._table[i][1]

    def __setitem__(self, k, v):
        # M[k] = v  (inserta o reemplaza)
        i = self._buscar(k)
        if i == -1:
            self._table.append([k,v])
        else:
            self._table[i][1] = v

    def __delitem__(self, k):
        # del M[k]  (KeyError si no existe)
        i = self._buscar(k)
        if i == -1:
            raise KeyError(k)
        self._table.pop(i)

    def __contains__(self, k):
        # k in M
        return self._buscar(k) != -1

    def __iter__(self):
        # for k in M  (genera las claves)
        for k, v in self._table:
            yield k

    def __eq__(self, otro):
        # M == otro  (mismos pares, sin importar el orden)
        if len(self) != len(otro):
            return False
        for k, v in self._table:
            if k not in otro or otro[k] != v:
                return False
        return True

    # ---------- dado ----------
    def __repr__(self):
        return '{' + ', '.join(f'{k!r}: {v!r}' for k, v in self._table) + '}'

## 5. Pruebas

Ejecuta cada celda. El comentario al lado de cada línea es la salida esperada, que es la misma que daría un `dict` de Python.

In [2]:
# --- Prueba 1: núcleo ---
M = UnsortedTableMap()
M['ana'] = 4.5
M['luis'] = 3.8
M['maria'] = 4.1
print(len(M))            # 3
print(M['luis'])         # 3.8

M['luis'] = 4.0          # reemplaza, NO agrega una entrada nueva
print(len(M))            # 3
print(M)                 # {'ana': 4.5, 'luis': 4.0, 'maria': 4.1}

del M['ana']
print(M._table)          # [['luis', 4.0], ['maria', 4.1]]
print('luis' in M, 'ana' in M)   # True False
print(list(M))           # ['luis', 'maria']

try:
    M['zoe']
except KeyError as e:
    print('KeyError:', e)    # KeyError: 'zoe'

try:
    del M['zoe']
except KeyError as e:
    print('KeyError:', e)    # KeyError: 'zoe'

3
3.8
3
{'ana': 4.5, 'luis': 4.0, 'maria': 4.1}
[['luis', 4.0], ['maria', 4.1]]
True False
['luis', 'maria']
KeyError: 'zoe'
KeyError: 'zoe'


In [3]:
# --- Prueba 2: igualdad (==) ---
A = UnsortedTableMap()
A['a'] = 1
A['b'] = 2
A['c'] = 3

B = UnsortedTableMap()
B['c'] = 3
B['b'] = 2
B['a'] = 1
print(A == B)          # True: el orden no importa

B['c'] = 4
print(A == B)          # False: mismo tamaño, distinto valor

del B['c']
print(A == B)          # False: distinto tamaño

print(A == {'a': 1, 'b': 2, 'c': 3})   # True: también se compara con un dict

True
False
False
True


In [4]:
# --- Prueba 3: contar frecuencias de palabras con TU diccionario ---
texto = 'el sol y la luna y el mar y el sol'
freq = UnsortedTableMap()
for palabra in texto.split():
    if palabra in freq:
        freq[palabra] = freq[palabra] + 1
    else:
        freq[palabra] = 1
print(freq)     # {'el': 3, 'sol': 2, 'y': 3, 'la': 1, 'luna': 1, 'mar': 1}

for palabra in freq:
    print(palabra, freq[palabra])

{'el': 3, 'sol': 2, 'y': 3, 'la': 1, 'luna': 1, 'mar': 1}
el 3
sol 2
y 3
la 1
luna 1
mar 1
